# Topic Modeling над колонкой `topic` в `d0rj/dialogsum-ru`

Цель: построить и сравнить тематические модели по полю `topic` датасета `d0rj/dialogsum-ru`.
Используем только классические подходы: **LDA** (на счётчиках слов) и **NMF** (на TF-IDF).

- Перебираем число тем `k` в диапазоне `[20, 200]` с шагом `10`.
- Без сэмплирования: используем весь объединённый датасет (train + validation + test).
- Для каждой модели и каждого `k` считаем когерентность `c_v` (gensim), для LDA — также перплексию, а также статистики размеров кластеров.
- Выбираем лучшую модель и лучшее `k` по когерентности с фильтрами на разумность кластеров.
- Делаем итоговое назначение темы каждому документу для дальнейшего использования.

Среда: рассчитано на запуск в Colab / Perplexity Compute с A100 high-memory.
BERTopic в этой версии не используется.

## Cell 1: Установка зависимостей

In [ ]:
# cell 1: installs
# При необходимости раскомментируйте установку нужных пакетов.
# !pip install -q pandas numpy pyarrow matplotlib seaborn scikit-learn gensim huggingface_hub fsspec

## Cell 2: Импорты и настройки

In [ ]:
# cell 2: imports and settings
import re
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

warnings.filterwarnings("ignore")

print("Imports OK. RANDOM_STATE =", RANDOM_STATE)

## Cell 3: Загрузка оригинальных сплитов

In [ ]:
# cell 3: load original splits
splits = {
    "train": "data/train-00000-of-00001-bcc43b46acda4001.parquet",
    "validation": "data/validation-00000-of-00001-7e263d81db1c7a12.parquet",
    "test": "data/test-00000-of-00001-2f13615b955ea947.parquet",
}
base_path = "hf://datasets/d0rj/dialogsum-ru/"

df_train = pd.read_parquet(base_path + splits["train"])
df_val = pd.read_parquet(base_path + splits["validation"])
df_test = pd.read_parquet(base_path + splits["test"])

for name, df in [("train", df_train), ("validation", df_val), ("test", df_test)]:
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.head(3))

## Cell 4: Объединение и базовая инспекция

In [ ]:
# cell 4: combine and inspect data
df_train = df_train.copy()
df_val = df_val.copy()
df_test = df_test.copy()
df_train["split"] = "train"
df_val["split"] = "validation"
df_test["split"] = "test"

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
print("df_all shape:", df_all.shape)
print("unique id:", df_all["id"].nunique() if "id" in df_all.columns else "<no id column>")

for col in ["id", "dialogue", "summary", "topic"]:
    if col in df_all.columns:
        missing = df_all[col].isna().sum()
        print(f"missing in {col}: {missing}")
    else:
        print(f"column '{col}' is not in df_all")

print("\nDistribution by split:")
print(df_all["split"].value_counts())

print("\nRandom topic examples:")
display(df_all[["split", "topic"]].sample(10, random_state=RANDOM_STATE))

## Cell 5: Предобработка `topic`

In [ ]:
# cell 5: preprocess topic texts
_clean_re = re.compile(r"[^\w\s]+", flags=re.UNICODE)
_space_re = re.compile(r"\s+")

def clean_topic(text):
    """Lowercase, remove special chars, collapse spaces."""
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = _clean_re.sub(" ", t)
    t = _space_re.sub(" ", t).strip()
    return t

df_all["topic_clean"] = df_all["topic"].apply(clean_topic)

empty_count = int((df_all["topic_clean"].str.len() == 0).sum())
short_count = int(((df_all["topic_clean"].str.split().str.len() < 2)
                   & (df_all["topic_clean"].str.len() > 0)).sum())
print(f"empty topic_clean: {empty_count}")
print(f"single-token topic_clean: {short_count}")

if empty_count > 0:
    df_model = df_all[df_all["topic_clean"].str.len() > 0].copy().reset_index(drop=True)
    print(f"removed {len(df_all) - len(df_model)} empty rows")
else:
    df_model = df_all.copy().reset_index(drop=True)
    print("no rows filtered")

print("df_model shape:", df_model.shape)
display(df_model[["topic", "topic_clean"]].sample(10, random_state=RANDOM_STATE))

## Cell 6: Токенизированный корпус для подсчёта когерентности

In [ ]:
# cell 6: prepare tokenized corpus for coherence
texts_tokenized = df_model["topic_clean"].str.split().tolist()

non_empty_mask = [len(toks) > 0 for toks in texts_tokenized]
if not all(non_empty_mask):
    n_before = len(df_model)
    df_model = df_model[non_empty_mask].copy().reset_index(drop=True)
    texts_tokenized = df_model["topic_clean"].str.split().tolist()
    print(f"removed {n_before - len(df_model)} empty-token docs; df_model shape: {df_model.shape}")
else:
    print("all docs have at least one token")

dictionary = Dictionary(texts_tokenized)
print("dictionary size (tokens):", len(dictionary))
print("num_docs:", dictionary.num_docs)
print("num_pos:", dictionary.num_pos)
print("avg tokens per doc:", float(np.mean([len(t) for t in texts_tokenized])))

## Cell 7: Векторизация (CountVectorizer для LDA, TfidfVectorizer для NMF)

In [ ]:
# cell 7: vectorizers
count_vec = CountVectorizer(min_df=5, max_df=0.9, max_features=5000)
tfidf_vec = TfidfVectorizer(min_df=5, max_df=0.9, max_features=5000)

X_lda = count_vec.fit_transform(df_model["topic_clean"].tolist())
X_tfidf = tfidf_vec.fit_transform(df_model["topic_clean"].tolist())

feature_names_count = np.array(count_vec.get_feature_names_out())
feature_names_tfidf = np.array(tfidf_vec.get_feature_names_out())

print("X_lda (counts) shape:", X_lda.shape)
print("X_tfidf shape:", X_tfidf.shape)
print("count vocab size:", len(feature_names_count))
print("tfidf vocab size:", len(feature_names_tfidf))

## Cell 8: Вспомогательные функции для тематических моделей

In [ ]:
# cell 8: helper functions for topic models
def get_top_words_sklearn(model, feature_names, n_top_words=10):
    """Top words per topic for sklearn LDA/NMF."""
    topics_words = []
    components = model.components_
    for topic_idx in range(components.shape[0]):
        top_idx = components[topic_idx].argsort()[::-1][:n_top_words]
        topics_words.append([feature_names[i] for i in top_idx])
    return topics_words


def get_doc_topic_assignments(doc_topic_distribution):
    """Return dominant topic per doc and the corresponding score."""
    arr = np.asarray(doc_topic_distribution)
    dominant = arr.argmax(axis=1)
    scores = arr.max(axis=1)
    return dominant, scores


def compute_cluster_stats(assignments, k=None):
    """min/max/mean cluster size and number of non-empty clusters."""
    assignments = np.asarray(assignments)
    if assignments.size == 0:
        return {
            "min_cluster_size": 0,
            "max_cluster_size": 0,
            "mean_cluster_size": 0.0,
            "n_nonempty_clusters": 0,
        }
    counts = pd.Series(assignments).value_counts()
    n_nonempty = int((counts > 0).sum())
    if k is not None:
        full_counts = np.zeros(k, dtype=int)
        for idx, c in counts.items():
            i = int(idx)
            if 0 <= i < k:
                full_counts[i] = int(c)
        return {
            "min_cluster_size": int(full_counts.min()),
            "max_cluster_size": int(full_counts.max()),
            "mean_cluster_size": float(full_counts.mean()),
            "n_nonempty_clusters": n_nonempty,
        }
    return {
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
        "mean_cluster_size": float(counts.mean()),
        "n_nonempty_clusters": n_nonempty,
    }


def compute_coherence(topics_words, texts_tokenized, dictionary):
    """c_v coherence via gensim, robust to errors."""
    try:
        topics_words_filtered = []
        for words in topics_words:
            kept = [w for w in words if w in dictionary.token2id]
            if len(kept) >= 2:
                topics_words_filtered.append(kept)
        if len(topics_words_filtered) < 2:
            return np.nan
        cm = CoherenceModel(
            topics=topics_words_filtered,
            texts=texts_tokenized,
            dictionary=dictionary,
            coherence="c_v",
        )
        return float(cm.get_coherence())
    except Exception as e:
        print(f"  [coherence error] {type(e).__name__}: {e}")
        return np.nan


def evaluate_lda_for_k(k, X_lda, feature_names, texts_tokenized, dictionary):
    """Fit LDA, compute perplexity, coherence, cluster stats."""
    res = {
        "model": "LDA",
        "k": k,
        "coherence_cv": np.nan,
        "perplexity": np.nan,
        "min_cluster_size": np.nan,
        "max_cluster_size": np.nan,
        "mean_cluster_size": np.nan,
        "n_nonempty_clusters": np.nan,
        "error": "",
    }
    try:
        lda = LatentDirichletAllocation(
            n_components=k,
            random_state=RANDOM_STATE,
            learning_method="online",
            max_iter=20,
            batch_size=4096,
            n_jobs=-1,
        )
        lda.fit(X_lda)
        try:
            res["perplexity"] = float(lda.perplexity(X_lda))
        except Exception as e:
            res["perplexity"] = np.nan
            res["error"] += f"perplexity:{type(e).__name__};"
        topics_words = get_top_words_sklearn(lda, feature_names, n_top_words=10)
        res["coherence_cv"] = compute_coherence(topics_words, texts_tokenized, dictionary)
        doc_topic = lda.transform(X_lda)
        assignments, _ = get_doc_topic_assignments(doc_topic)
        stats = compute_cluster_stats(assignments, k=k)
        res.update(stats)
    except Exception as e:
        res["error"] += f"fit:{type(e).__name__}:{e};"
    return res


def evaluate_nmf_for_k(k, X_tfidf, feature_names, texts_tokenized, dictionary):
    """Fit NMF, compute coherence and cluster stats."""
    res = {
        "model": "NMF",
        "k": k,
        "coherence_cv": np.nan,
        "perplexity": np.nan,
        "min_cluster_size": np.nan,
        "max_cluster_size": np.nan,
        "mean_cluster_size": np.nan,
        "n_nonempty_clusters": np.nan,
        "error": "",
    }
    try:
        nmf = NMF(
            n_components=k,
            random_state=RANDOM_STATE,
            init="nndsvd",
            max_iter=500,
            beta_loss="frobenius",
            solver="cd",
        )
        W = nmf.fit_transform(X_tfidf)
        topics_words = get_top_words_sklearn(nmf, feature_names, n_top_words=10)
        res["coherence_cv"] = compute_coherence(topics_words, texts_tokenized, dictionary)
        assignments, _ = get_doc_topic_assignments(W)
        stats = compute_cluster_stats(assignments, k=k)
        res.update(stats)
    except Exception as e:
        res["error"] += f"fit:{type(e).__name__}:{e};"
    return res

print("Helper functions defined.")

## Cell 9: Полный перебор LDA и NMF по k

Внимание: полный сметоп `k ∈ [20, 200]` с шагом 10 на полном корпусе может занимать значительное время даже на A100 high-memory.
Используется весь датасет без сэмплирования.

In [ ]:
# cell 9: sweep LDA and NMF over k
k_list = list(range(20, 201, 10))
print("k_list:", k_list)
print("Total fits:", 2 * len(k_list), "( LDA + NMF for every k )")

results = []
for i, k in enumerate(k_list, start=1):
    print(f"\n[{i}/{len(k_list)}] k={k} | LDA fit ...")
    res_lda = evaluate_lda_for_k(k, X_lda, feature_names_count, texts_tokenized, dictionary)
    coh = res_lda["coherence_cv"]
    coh_str = f"{coh:.4f}" if isinstance(coh, float) and not np.isnan(coh) else "NaN"
    print(
        f"  LDA k={k}: coh={coh_str}",
        f"perpl={res_lda['perplexity']}",
        f"min={res_lda['min_cluster_size']} max={res_lda['max_cluster_size']} nonempty={res_lda['n_nonempty_clusters']}",
    )
    results.append(res_lda)

    print(f"[{i}/{len(k_list)}] k={k} | NMF fit ...")
    res_nmf = evaluate_nmf_for_k(k, X_tfidf, feature_names_tfidf, texts_tokenized, dictionary)
    coh = res_nmf["coherence_cv"]
    coh_str = f"{coh:.4f}" if isinstance(coh, float) and not np.isnan(coh) else "NaN"
    print(
        f"  NMF k={k}: coh={coh_str}",
        f"min={res_nmf['min_cluster_size']} max={res_nmf['max_cluster_size']} nonempty={res_nmf['n_nonempty_clusters']}",
    )
    results.append(res_nmf)

results_df = pd.DataFrame(results, columns=[
    "model", "k", "coherence_cv", "perplexity",
    "min_cluster_size", "max_cluster_size", "mean_cluster_size",
    "n_nonempty_clusters", "error",
])
print("\nSweep finished. results_df shape:", results_df.shape)

## Cell 10: Таблицы и графики результатов

In [ ]:
# cell 10: results table and plots
display(results_df)

print("\nTop-5 by coherence_cv:")
display(results_df.sort_values("coherence_cv", ascending=False).head(5))

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=results_df, x="k", y="coherence_cv", hue="model", marker="o", ax=ax)
ax.set_title("Когерентность c_v по числу тем k")
ax.set_xlabel("k (число тем)")
ax.set_ylabel("coherence c_v")
ax.legend(title="model")
plt.tight_layout()
plt.show()

lda_df = results_df[results_df["model"] == "LDA"]
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=lda_df, x="k", y="perplexity", marker="o", color="C2", ax=ax)
ax.set_title("LDA: перплексия по k")
ax.set_xlabel("k")
ax.set_ylabel("perplexity")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
sns.lineplot(data=results_df, x="k", y="min_cluster_size", hue="model", marker="o", ax=axes[0])
axes[0].set_title("Минимальный размер кластера по k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("min_cluster_size")
sns.lineplot(data=results_df, x="k", y="max_cluster_size", hue="model", marker="o", ax=axes[1])
axes[1].set_title("Максимальный размер кластера по k")
axes[1].set_xlabel("k")
axes[1].set_ylabel("max_cluster_size")
plt.tight_layout()
plt.show()

## Cell 11: Выбор лучшей модели и k

In [ ]:
# cell 11: choose best model and k
n_docs = len(df_model)
max_cluster_cap = 0.4 * n_docs
min_cluster_floor = 20

valid_df = results_df.dropna(subset=["coherence_cv"]).copy()
valid_df = valid_df[
    (valid_df["min_cluster_size"] >= min_cluster_floor)
    & (valid_df["max_cluster_size"] <= max_cluster_cap)
]

if len(valid_df) == 0:
    print("WARNING: no configuration satisfies the cluster-size filters — falling back to global max coherence_cv.")
    valid_df = results_df.dropna(subset=["coherence_cv"]).copy()

best_row = valid_df.sort_values("coherence_cv", ascending=False).iloc[0]
best_model_name = best_row["model"]
best_k = int(best_row["k"])
best_coherence = float(best_row["coherence_cv"])

lda_valid = valid_df[valid_df["model"] == "LDA"].sort_values("coherence_cv", ascending=False)
nmf_valid = valid_df[valid_df["model"] == "NMF"].sort_values("coherence_cv", ascending=False)

if len(lda_valid) > 0:
    k_best_lda = int(lda_valid.iloc[0]["k"])
    coh_best_lda = float(lda_valid.iloc[0]["coherence_cv"])
else:
    lda_any = results_df[results_df["model"] == "LDA"].dropna(subset=["coherence_cv"]).sort_values("coherence_cv", ascending=False)
    k_best_lda = int(lda_any.iloc[0]["k"]) if len(lda_any) else None
    coh_best_lda = float(lda_any.iloc[0]["coherence_cv"]) if len(lda_any) else np.nan

if len(nmf_valid) > 0:
    k_best_nmf = int(nmf_valid.iloc[0]["k"])
    coh_best_nmf = float(nmf_valid.iloc[0]["coherence_cv"])
else:
    nmf_any = results_df[results_df["model"] == "NMF"].dropna(subset=["coherence_cv"]).sort_values("coherence_cv", ascending=False)
    k_best_nmf = int(nmf_any.iloc[0]["k"]) if len(nmf_any) else None
    coh_best_nmf = float(nmf_any.iloc[0]["coherence_cv"]) if len(nmf_any) else np.nan

print("=== Selection results ===")
print(f"best_model_name = {best_model_name}")
print(f"best_k          = {best_k}")
print(f"best_coherence  = {best_coherence:.4f}")
print("--- LDA best ---")
print(f"k_best_lda      = {k_best_lda}")
print(f"coh_best_lda    = {coh_best_lda}")
print("--- NMF best ---")
print(f"k_best_nmf      = {k_best_nmf}")
print(f"coh_best_nmf    = {coh_best_nmf}")
print("\nbest row stats:")
display(best_row.to_frame().T)

## Cell 12: Обучение финальных моделей

In [ ]:
# cell 12: fit final models
lda_final = None
lda_doc_topic = None
if k_best_lda is not None:
    print(f"Fitting final LDA with k={k_best_lda} ...")
    lda_final = LatentDirichletAllocation(
        n_components=k_best_lda,
        random_state=RANDOM_STATE,
        learning_method="online",
        max_iter=30,
        batch_size=4096,
        n_jobs=-1,
    )
    lda_doc_topic = lda_final.fit_transform(X_lda)
    print("LDA done.")

nmf_final = None
nmf_doc_topic = None
if k_best_nmf is not None:
    print(f"Fitting final NMF with k={k_best_nmf} ...")
    nmf_final = NMF(
        n_components=k_best_nmf,
        random_state=RANDOM_STATE,
        init="nndsvd",
        max_iter=500,
        beta_loss="frobenius",
        solver="cd",
    )
    nmf_doc_topic = nmf_final.fit_transform(X_tfidf)
    print("NMF done.")

if lda_doc_topic is not None:
    lda_assign, lda_score = get_doc_topic_assignments(lda_doc_topic)
    df_model["lda_topic"] = lda_assign
    df_model["lda_topic_score"] = lda_score
if nmf_doc_topic is not None:
    nmf_assign, nmf_score = get_doc_topic_assignments(nmf_doc_topic)
    df_model["nmf_topic"] = nmf_assign
    df_model["nmf_topic_score"] = nmf_score

if best_model_name == "LDA" and lda_doc_topic is not None:
    df_model["best_topic"] = df_model["lda_topic"]
    df_model["best_topic_score"] = df_model["lda_topic_score"]
elif best_model_name == "NMF" and nmf_doc_topic is not None:
    df_model["best_topic"] = df_model["nmf_topic"]
    df_model["best_topic_score"] = df_model["nmf_topic_score"]

if "id" in df_model.columns and "id" in df_all.columns:
    merge_cols = [c for c in ["id", "lda_topic", "lda_topic_score", "nmf_topic", "nmf_topic_score", "best_topic", "best_topic_score"] if c in df_model.columns]
    df_all = df_all.merge(df_model[merge_cols], on="id", how="left")
    print("Merged topic assignments back into df_all. df_all shape:", df_all.shape)

print("\ndf_model columns:", list(df_model.columns))

## Cell 13: Инспекция итоговых тем и примеры

In [ ]:
# cell 13: inspect final topics and examples
lda_topics_words = None
nmf_topics_words = None

if lda_final is not None:
    print(f"=== Top words: final LDA (k={k_best_lda}) ===")
    lda_topics_words = get_top_words_sklearn(lda_final, feature_names_count, n_top_words=10)
    for i, words in enumerate(lda_topics_words):
        print(f"LDA topic {i}: {', '.join(words)}")

if nmf_final is not None:
    print(f"\n=== Top words: final NMF (k={k_best_nmf}) ===")
    nmf_topics_words = get_top_words_sklearn(nmf_final, feature_names_tfidf, n_top_words=10)
    for i, words in enumerate(nmf_topics_words):
        print(f"NMF topic {i}: {', '.join(words)}")

if "best_topic" in df_model.columns:
    print(f"\n=== Распределение по лучшей модели ({best_model_name}) ===")
    best_counts = df_model["best_topic"].value_counts().sort_values(ascending=False)
    print("Total topics with at least 1 doc:", int((best_counts > 0).sum()))
    display(best_counts.head(20).to_frame("n_docs"))

    best_topics_words = lda_topics_words if best_model_name == "LDA" else nmf_topics_words

    show_topics = list(best_counts.head(5).index) + list(best_counts.tail(5).index)
    show_topics = list(dict.fromkeys([int(t) for t in show_topics]))
    for t in show_topics:
        words = best_topics_words[t] if best_topics_words is not None and 0 <= t < len(best_topics_words) else []
        size = int((df_model["best_topic"] == t).sum())
        print(f"\n--- Topic {t} ({best_model_name}) | size={size} ---")
        print("  top words:", ", ".join(words))
        sub = df_model[df_model["best_topic"] == t].head(8)
        for _, row in sub.iterrows():
            print("   *", str(row["topic"])[:200])

    print("\n=== Examples (random) ===")
    cols = [c for c in ["topic", "topic_clean", "best_topic", "best_topic_score"] if c in df_model.columns]
    display(df_model[cols].sample(min(20, len(df_model)), random_state=RANDOM_STATE))

## Cell 14: Итоговые выводы

В этом ноутбуке мы:

- загрузили все три сплита `d0rj/dialogsum-ru` через `pandas.read_parquet` напрямую с Hugging Face и объединили их в `df_all` (без сэмплирования);
- предобработали колонку `topic` (`topic_clean`) — нижний регистр, очистка спецсимволов, нормализация пробелов;
- построили `CountVectorizer` для LDA и `TfidfVectorizer` для NMF, а также общий словарь gensim для подсчёта когерентности `c_v`;
- провели полный сметоп по числу тем `k ∈ [20, 200]` с шагом `10` для двух моделей: **LDA** и **NMF**;
- собрали единую таблицу `results_df` с когерентностью, перплексией (для LDA) и статистиками размеров кластеров;
- выбрали лучшую модель и лучшее `k` по максимуму `c_v` с фильтрами `min_cluster_size >= 20` и `max_cluster_size <= 40% * n_docs`; конкретные значения `best_model_name`, `best_k`, `best_coherence`, `k_best_lda`, `k_best_nmf` напечатаны в ячейках выше — на них и нужно ориентироваться, а не на фиксированные числа в этом тексте;
- обучили финальные модели и записали назначения тем в `df_model` (`lda_topic`, `nmf_topic`, `best_topic` и соответствующие скоры), при наличии `id` — смержили обратно в `df_all`.

**Почему важна когерентность.** `c_v` оценивает интерпретируемость тем — насколько топ-слова темы статистически сочетаются в реальных документах. Это более содержательный сигнал, чем чисто статистическая перплексия LDA, и поэтому используется как основной критерий выбора `k`.

**Почему важна структура кластеров.** Большие однотипные модели любят коллапсировать в одну гигантскую тему-«помойку» или, наоборот, плодить десятки почти пустых тем. Фильтры по `min_cluster_size` и `max_cluster_size` отсекают такие вырожденные конфигурации до сравнения по когерентности.

**Как использовать итоговые темы дальше.** Полученное распределение `best_topic` можно применять:

- для **группировки похожих тем** при разметке и анализе диалогов;
- для **консолидации редких категорий** (`topic` с малой частотой) в более крупные семантические кластеры;
- для **стратификации** при сплите train/val/test и при сэмплировании для аннотации, чтобы избегать перекоса по темам;
- как **слабые метки тем** в downstream-моделях распознавания намерений (вспомогательный сигнал, не таргет).

Конкретные числовые выводы — лучшая модель, лучшее `k`, значение когерентности, размеры топ-кластеров — приведены в выводе ячеек 10–13 и в таблице `results_df`.